# Bootstrap Resampling For EigenBench Error Bars

This notebook loads a run spec from `runs/`, reads that run's evaluation transcripts, bootstrap-resamples the extracted comparisons, retrains the selected BT/BTD model on each sample, and summarizes EigenTrust/Elo uncertainty with standard deviations and percentile confidence intervals.


In [ ]:
from __future__ import annotations

import json
import math
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

sys.path.append('..')

from pipeline.config import load_run_spec
from pipeline.utils import (
    extract_comparisons_with_ties_criteria,
    handle_inconsistencies_with_ties_criteria,
    load_records,
)
from pipeline.train import (
    Comparisons,
    CriteriaComparisons,
    CriteriaVectorBTD,
    VectorBT,
    build_model_labels,
    eigentrust_to_elo,
    train_vector_bt,
)
from pipeline.trust import compute_trust_matrix, compute_trust_matrix_ties, eigentrust, row_normalize


In [ ]:
# User parameters
SPEC_PATH = "../runs/my_run/spec.py"

# Optional override. Leave as None to use the run's collection.evaluations_path.
EVALUATIONS_PATH_OVERRIDE = None

N_BOOTSTRAPS = 100
RANDOM_SEED = 42
SAVE_BOOTSTRAP_MODELS = False
SAVE_EACH_TRUST_MATRIX = True
OUTPUT_SUBDIR = "bootstrap"
OUTPUT_BASENAME = "samples.json"

# Training overrides. Set to None to inherit from the run spec.
MODEL_KIND = None          # None | "btd_ties" | "bt"
DIM = None                 # None -> use the first entry from training.dims
LR = None
WEIGHT_DECAY = None
MAX_EPOCHS = None
BATCH_SIZE = None
DEVICE = None
SEPARATE_CRITERIA = None   # None -> inherit from training.separate_criteria
NORMALIZE = False          # only relevant for BT models exposing v embeddings


In [ ]:
spec, run_dir = load_run_spec(SPEC_PATH)
train_cfg = spec.get("training", {})
constitution_cfg = spec.get("constitution", {})
collection_cfg = spec.get("collection", {})

model_kind = MODEL_KIND or train_cfg.get("model", "btd_ties")
dim = int(DIM if DIM is not None else list(train_cfg.get("dims", [2]))[0])
lr = float(LR if LR is not None else train_cfg.get("lr", 1e-3))
weight_decay = float(WEIGHT_DECAY if WEIGHT_DECAY is not None else train_cfg.get("weight_decay", 0.0))
max_epochs = int(MAX_EPOCHS if MAX_EPOCHS is not None else train_cfg.get("max_epochs", 1000))
batch_size = int(BATCH_SIZE if BATCH_SIZE is not None else train_cfg.get("batch_size", 32))
device = DEVICE or train_cfg.get("device", "cpu")
separate_criteria = bool(
    train_cfg.get("separate_criteria", False)
    if SEPARATE_CRITERIA is None
    else SEPARATE_CRITERIA
)
num_criteria = int(constitution_cfg["num_criteria"])

if EVALUATIONS_PATH_OVERRIDE:
    evaluations_path = Path(EVALUATIONS_PATH_OVERRIDE).expanduser().resolve()
else:
    evaluations_path = Path(collection_cfg["evaluations_path"]).expanduser().resolve()

output_dir = Path(run_dir) / OUTPUT_SUBDIR
output_dir.mkdir(parents=True, exist_ok=True)
models_dir = output_dir / "models"
if SAVE_BOOTSTRAP_MODELS:
    models_dir.mkdir(parents=True, exist_ok=True)

print(f"Run dir: {run_dir}")
print(f"Evaluations path: {evaluations_path}")
print(f"Output dir: {output_dir}")
print(f"Model kind: {model_kind}")
print(f"Dim: {dim}")
print(f"Separate criteria: {separate_criteria}")


In [ ]:
evaluations = load_records(evaluations_path)
comparisons, _, extracted_name_map = extract_comparisons_with_ties_criteria(
    evaluations,
    num_criteria=num_criteria,
    verbose=True,
    return_name_map=True,
)
comparisons = handle_inconsistencies_with_ties_criteria(comparisons)

if not separate_criteria:
    comparisons = [[0] + row[1:] for row in comparisons]

if not comparisons:
    raise ValueError("No usable comparisons were extracted from the evaluations file.")

num_models = len(set([row[2] for row in comparisons] + [row[3] for row in comparisons] + [row[4] for row in comparisons]))
num_criteria_eff = len(set(row[0] for row in comparisons))

model_labels = build_model_labels(num_models, spec.get("models", {}), extracted_name_map)

print(f"Evaluations loaded: {len(evaluations)}")
print(f"Comparisons extracted: {len(comparisons)}")
print(f"Number of models: {num_models}")
print(f"Effective criteria: {num_criteria_eff}")
print("Model labels:")
for idx, label in enumerate(model_labels):
    print(f"  [{idx}] {label}")


In [ ]:
def build_model_and_loader(sampled_comparisons):
    if model_kind == "btd_ties":
        model = CriteriaVectorBTD(num_criteria_eff, num_models, dim)
        dataloader = DataLoader(CriteriaComparisons(sampled_comparisons), batch_size=batch_size, shuffle=True)
        return model, dataloader, True, True

    if model_kind == "bt":
        flattened = [[0] + row[1:] for row in sampled_comparisons]
        model = VectorBT(num_models, dim)
        dataloader = DataLoader(Comparisons(flattened), batch_size=batch_size, shuffle=True)
        return model, dataloader, False, False

    raise ValueError(f"Unsupported MODEL_KIND: {model_kind}")


def train_bootstrap_sample(sampled_comparisons):
    model, dataloader, use_btd, criterion_mode = build_model_and_loader(sampled_comparisons)
    train_vector_bt(
        model=model,
        dataloader=dataloader,
        lr=lr,
        weight_decay=weight_decay,
        max_epochs=max_epochs,
        device=device,
        save_path=None,
        normalize=NORMALIZE,
        use_btd=use_btd,
        criterion_mode=criterion_mode,
        verbose=False,
    )

    if use_btd:
        trust_matrix = compute_trust_matrix_ties(model, device=device)
        trust_vector = eigentrust(trust_matrix, alpha=0, verbose=False)
    else:
        score_matrix = compute_trust_matrix(model, device=device)
        trust_matrix = row_normalize(score_matrix)
        trust_vector = eigentrust(trust_matrix, alpha=0, verbose=False)

    return trust_matrix.detach().cpu().numpy(), trust_vector.detach().cpu().numpy(), model


In [ ]:
rng = random.Random(RANDOM_SEED)
bootstrap_records = []
trust_vectors = []
elo_vectors = []

for sample_idx in tqdm(range(N_BOOTSTRAPS), desc="Bootstrap samples"):
    sampled_comparisons = [comparisons[rng.randrange(len(comparisons))] for _ in range(len(comparisons))]
    trust_matrix, trust_vector, model = train_bootstrap_sample(sampled_comparisons)
    elo_vector = eigentrust_to_elo(trust_vector, num_models)

    record = {
        "sample_idx": sample_idx,
        "trust_vector": trust_vector.tolist(),
        "elo_vector": elo_vector.tolist(),
    }
    if SAVE_EACH_TRUST_MATRIX:
        record["trust_matrix"] = trust_matrix.tolist()
    bootstrap_records.append(record)
    trust_vectors.append(trust_vector)
    elo_vectors.append(elo_vector)

    if SAVE_BOOTSTRAP_MODELS:
        torch.save(model.state_dict(), models_dir / f"model_{sample_idx:04d}.pt")

output_path = output_dir / OUTPUT_BASENAME
with output_path.open("w", encoding="utf-8") as f:
    json.dump(bootstrap_records, f, indent=2)

trust_vectors = np.asarray(trust_vectors, dtype=float)
elo_vectors = np.asarray(elo_vectors, dtype=float)

print(f"Saved bootstrap samples to {output_path}")
print(f"Trust vector array shape: {trust_vectors.shape}")
print(f"Elo vector array shape: {elo_vectors.shape}")


In [ ]:
elo_means = np.mean(elo_vectors, axis=0)
elo_std = np.std(elo_vectors, axis=0, ddof=1)
elo_lower = np.percentile(elo_vectors, 2.5, axis=0)
elo_upper = np.percentile(elo_vectors, 97.5, axis=0)

summary_rows = []
for idx, label in enumerate(model_labels):
    row = {
        "model_index": idx,
        "model_name": label,
        "elo_mean": float(elo_means[idx]),
        "elo_std": float(elo_std[idx]),
        "elo_ci_lower": float(elo_lower[idx]),
        "elo_ci_upper": float(elo_upper[idx]),
    }
    summary_rows.append(row)

summary_rows = sorted(summary_rows, key=lambda row: row["elo_mean"], reverse=True)

summary_path = output_dir / "summary.json"
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary_rows, f, indent=2)

for row in summary_rows:
    print(
        f"[{row['model_index']:>2}] {row['model_name']}: "
        f"mean={row['elo_mean']:.2f}, std={row['elo_std']:.2f}, "
        f"95% CI=[{row['elo_ci_lower']:.2f}, {row['elo_ci_upper']:.2f}]"
    )

print(f"Saved summary to {summary_path}")


In [ ]:
ordered_labels = [row["model_name"] for row in summary_rows]
ordered_means = np.array([row["elo_mean"] for row in summary_rows])
ordered_lower = np.array([row["elo_ci_lower"] for row in summary_rows])
ordered_upper = np.array([row["elo_ci_upper"] for row in summary_rows])

x = np.arange(len(summary_rows))
yerr = np.vstack([ordered_means - ordered_lower, ordered_upper - ordered_means])

plt.figure(figsize=(max(10, len(summary_rows) * 0.45), 6))
plt.errorbar(x, ordered_means, yerr=yerr, fmt='o', capsize=4)
plt.xticks(x, ordered_labels, rotation=45, ha='right')
plt.ylabel('EigenBench Elo')
plt.title('Bootstrap Elo Means with 95% Confidence Intervals')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
